# Chưng cất tri thức — YOLO26s dạy YOLO26n

Trả lời một câu hỏi duy nhất: **có thể làm mô hình nhẹ chính xác hơn mà không tăng một chút
chi phí suy luận nào không?** Mô hình chưng cất xuất ra có đúng số tham số, đúng tốc độ và
đúng dung lượng như mô hình huấn luyện thường — chỉ khác ở trọng số.

### Ba mô hình được huấn luyện

| Vai trò | Kiến trúc | Khác biệt |
|---|---|---|
| **Thầy** | `yolo26s` | Lớn hơn, dùng để hướng dẫn |
| **Trò đối chứng** | `yolo26n` | Huấn luyện thường |
| **Trò chưng cất** | `yolo26n` | **Chỉ khác đúng một tham số:** `distill_model` |

Hai mô hình trò dùng **cùng hạt giống, cùng dữ liệu, cùng số epoch, cùng bộ tối ưu, cùng lịch
học, cùng phép tăng cường**. Khác biệt duy nhất là chưng cất — đó là điều kiện để quy kết
chênh lệch quan sát được cho đúng nguyên nhân.

### Vì sao YOLO26 chứ không phải YOLOv8

Ultralytics **không hỗ trợ chưng cất chéo họ**. Thầy và trò bắt buộc cùng một thế hệ YOLO.
Mà `yolov8n` là mô hình nhỏ nhất trong họ của nó — không có trò nào nhỏ hơn để dạy. YOLO26 có
sẵn cặp `s → n` được tài liệu chính thức khuyến nghị.

### Năm cấu hình được đánh giá

Thầy · Trò đối chứng · Trò chưng cất · Trò chưng cất ONNX FP32 · Trò chưng cất ONNX INT8

Tất cả đo bằng **cùng một bộ `pycocotools`** trên **cùng tập kiểm tra**, để ghép thẳng vào
bảng so sánh của báo cáo hiện có.

---

## ⚠️ Quy trình hai lượt và ngân sách GPU

| Lượt | Cấu hình | Thời gian |
|---|---|---|
| Chạy thử | `RUN_MODE = "SMOKE_TEST"` | **~10 phút** |
| Huấn luyện thật | `RUN_MODE = "FULL_TRAIN"` + `CONFIRM_FULL_TRAIN = True` | **~3 giờ** |

Lượt thật tốn khoảng 1/10 quota tuần của Kaggle. Nếu phiên hay bị ngắt, dùng `RUN_STAGE` để
chia từng giai đoạn ra các phiên riêng.

---
## §0 · Cấu hình và môi trường

In [1]:
# ════════════════════════════ CẤU HÌNH ════════════════════════════
RUN_MODE = "FULL_TRAIN"        # "SMOKE_TEST" | "FULL_TRAIN"
CONFIRM_FULL_TRAIN = True     # phải đổi True khi chạy FULL_TRAIN
RUN_STAGE = "ALL"              # ALL | TEACHER | BASELINE | KD | EXPORT_EVAL

SEED = 42
IMGSZ = 640

TEACHER_MODEL = "yolo26s.pt"   # có thể đổi thành yolo26m.pt nếu dư GPU
STUDENT_MODEL = "yolo26n.pt"

EPOCHS_TEACHER = 50
EPOCHS_STUDENT = 50            # DÙNG CHUNG cho cả trò đối chứng và trò chưng cất
PATIENCE = 12
BATCH_TEACHER, BATCH_BASELINE, BATCH_KD = 16, 24, 8
DIS_WEIGHT = 6.0               # trọng số hàm mất mát chưng cất (mặc định Ultralytics)

CONF_EVAL, CONF_VIZ, IOU_MATCH = 0.001, 0.25, 0.50
N_CALIB = 300
N_SPEED_WARMUP, N_SPEED_RUNS = 10, 100

# Ngưỡng chấp nhận — kiểm tra tự động ở §6
THRESH = {
    "kd_not_worse_than_baseline": -0.005,   # KD không được kém hơn quá 0,5 điểm mAP50-95
    "int8_max_drop": 0.015,                 # INT8 giảm không quá 1,5 điểm so với KD FP32
    "int8_min_size_reduction": 0.50,        # INT8 phải nhỏ hơn FP32 ít nhất 50%
}
# ═══════════════════════════════════════════════════════════════════

import os, sys, json, time, random, shutil, subprocess, hashlib, warnings
from pathlib import Path

warnings.filterwarnings("ignore")

if RUN_MODE == "FULL_TRAIN" and not CONFIRM_FULL_TRAIN:
    raise RuntimeError(
        "\n" + "=" * 66 + "\n"
        "  Ban dang yeu cau FULL_TRAIN (~3 gio GPU).\n" + "=" * 66 + "\n"
        "  Hay chay SMOKE_TEST cho tron ven truoc, sau do dat:\n"
        "      CONFIRM_FULL_TRAIN = True\n"
        + "=" * 66)

SMOKE = RUN_MODE == "SMOKE_TEST"

if Path("/kaggle/working").exists():
    PLATFORM, WORK, INPUT_ROOTS = "kaggle", Path("/kaggle/working"), [Path("/kaggle/input")]
elif Path("/content").exists():
    PLATFORM, WORK = "colab", Path("/content/work")
    INPUT_ROOTS = [Path("/content/drive/MyDrive"), Path("/content")]
else:
    PLATFORM, WORK, INPUT_ROOTS = "local", Path.cwd() / "work", [Path.cwd()]

OUT = WORK / "kd_outputs" / ("smoke" if SMOKE else "full")
DATA = WORK / "data"
COCO = OUT / "01_dataset"
RUNS = OUT / "runs"
EXPORTS = OUT / "05_exports"
RESULTS = OUT / "06_results"
for d in (OUT, COCO, RUNS, EXPORTS, RESULTS,
          RESULTS / "metrics", RESULTS / "plots", RESULTS / "tables"):
    d.mkdir(parents=True, exist_ok=True)

STAGE_DIR = {s: OUT / f"stage_{s.lower()}" for s in ("TEACHER", "BASELINE", "KD")}
for d in STAGE_DIR.values():
    d.mkdir(parents=True, exist_ok=True)

SMOKE_N = {"train": 48, "valid": 16, "test": 16}
if SMOKE:
    EPOCHS_TEACHER = EPOCHS_STUDENT = 1
    BATCH_TEACHER = BATCH_BASELINE = BATCH_KD = 2
    IMGSZ, N_CALIB, N_SPEED_WARMUP, N_SPEED_RUNS = 160, 32, 2, 5
    print("*" * 66)
    print("*  CHE DO CHAY THU — so lieu KHONG dung de bao cao")
    print("*  Muc tieu: xac nhan ca 3 giai doan train, export va danh gia deu chay")
    print("*" * 66)
else:
    print("=" * 66)
    print("  CHE DO HUAN LUYEN THAT — uoc tinh ~3 gio tren Tesla T4")
    print(f"  thay {EPOCHS_TEACHER} epoch | tro {EPOCHS_STUDENT} epoch x 2")
    print("=" * 66)

print(f"\nnen tang    : {PLATFORM}")
print(f"giai doan   : {RUN_STAGE}")
print(f"thu muc ra  : {OUT}")


def stage_enabled(s):
    return RUN_STAGE in ("ALL", s)


def stage_done(s):
    return (STAGE_DIR[s] / "stage_completed.json").exists()


def mark_done(s, info):
    (STAGE_DIR[s] / "stage_completed.json").write_text(json.dumps(info, indent=2))

  CHE DO HUAN LUYEN THAT — uoc tinh ~3 gio tren Tesla T4
  thay 50 epoch | tro 50 epoch x 2

nen tang    : kaggle
giai doan   : ALL
thu muc ra  : /kaggle/working/kd_outputs/full


### 0.1 Kiểm tra GPU

Cell này tự đứng một mình được. Từ PyTorch 2.8, các bản build CUDA 12.8 đã bỏ hỗ trợ kiến
trúc Pascal — **Tesla P100 không dùng được**. Trên Kaggle chọn `GPU T4 x2`.

In [2]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "\n" + "=" * 66 + "\n  CHUA BAT GPU\n" + "=" * 66 + "\n"
        "  Kaggle : Settings -> Accelerator -> 'GPU T4 x2'\n"
        "  Colab  : Runtime -> Change runtime type -> 'T4 GPU'\n" + "=" * 66)

_cap = torch.cuda.get_device_capability(0)
_sm = f"sm_{_cap[0]}{_cap[1]}"
_archs = [a for a in torch.cuda.get_arch_list() if a.startswith("sm_")]
print(f"GPU          : {torch.cuda.get_device_name(0)}  ({_sm})")
print(f"VRAM         : {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB")
print(f"torch        : {torch.__version__}")
print(f"torch ho tro : {', '.join(_archs)}")
if _sm not in _archs:
    raise RuntimeError(
        f"GPU {_sm} khong duoc torch {torch.__version__} ho tro. Doi sang T4.")
_ = (torch.randn(64, 64, device="cuda") @ torch.randn(64, 64, device="cuda")).sum()
torch.cuda.synchronize()
print("GPU test     : OK")
DEVICE = 0

import multiprocessing
N_CPU = multiprocessing.cpu_count()

GPU          : Tesla T4  (sm_75)
VRAM         : 14.6 GB
torch        : 2.10.0+cu128
torch ho tro : sm_70, sm_75, sm_80, sm_86, sm_90, sm_100, sm_120
GPU test     : OK


### 0.2 Cài thư viện và **kiểm tra chưng cất có được hỗ trợ không**

Đây là bước chặn quan trọng. Nếu phiên bản Ultralytics không có tham số `distill_model`,
notebook phải dừng ngay với thông báo rõ ràng — **tuyệt đối không âm thầm chuyển sang một
cơ chế chưng cất tự viết**, vì như vậy kết quả sẽ không còn là "chưng cất chính thức của
Ultralytics" như báo cáo tuyên bố.

In [3]:
import torchvision

CONSTRAINTS = OUT / "pip-constraints.txt"
CONSTRAINTS.write_text(
    f"torch=={torch.__version__.split('+')[0]}\n"
    f"torchvision=={torchvision.__version__.split('+')[0]}\n")


def pip_install(args):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "-c", str(CONSTRAINTS)] + args.split(), check=False)


pip_install("-U ultralytics")
pip_install("onnx onnxruntime onnxslim")
pip_install("pycocotools")

import ultralytics
from ultralytics.cfg import DEFAULT_CFG_DICT

print("ultralytics :", ultralytics.__version__)

_missing = [k for k in ("distill_model", "dis") if k not in DEFAULT_CFG_DICT]
if _missing:
    raise RuntimeError(
        "\n" + "=" * 66 + "\n"
        "  PHIEN BAN ULTRALYTICS NAY KHONG HO TRO CHUNG CAT\n" + "=" * 66 + "\n"
        f"  Phien ban  : {ultralytics.__version__}\n"
        f"  Thieu tham so: {', '.join(_missing)}\n\n"
        "  Chung cat can ultralytics >= 8.4.x co tinh nang Knowledge Distillation.\n"
        "  Chay:  pip install -U ultralytics   roi Restart Session.\n"
        "  KHONG tu viet co che chung cat thay the — ket qua se khong con\n"
        "  la 'chung cat chinh thuc' nhu bao cao tuyen bo.\n" + "=" * 66)

print(f"chung cat   : ho tro (distill_model, dis)")
print(f"dis mac dinh: {DEFAULT_CFG_DICT.get('dis')}")

os.environ["YOLO_CONFIG_DIR"] = str(OUT / "00_env" / "yolo")
os.environ["MPLCONFIGDIR"] = str(OUT / "00_env" / "mpl")
Path(os.environ["YOLO_CONFIG_DIR"]).mkdir(parents=True, exist_ok=True)
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 75.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 238.4/238.4 kB 14.0 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
ultralytics : 8.4.108
chung cat   : ho tro (distill_model, dis)
dis mac dinh: 6.0


In [4]:
import numpy as np


def seed_everything(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)


seed_everything()
(OUT / "00_env" / "environment.json").write_text(json.dumps({
    "platform": PLATFORM, "torch": torch.__version__,
    "torchvision": torchvision.__version__, "ultralytics": ultralytics.__version__,
    "gpu": torch.cuda.get_device_name(0), "sm": _sm, "n_cpu": N_CPU,
    "run_mode": RUN_MODE, "seed": SEED,
}, indent=2))
print("seed =", SEED)

seed = 42


---
## §1 · Dữ liệu

### 1.1 Dò tìm và sao chép dataset

Không dùng đường dẫn cứng. Notebook quét đệ quy tìm thư mục chứa `data.yaml` kèm đủ các tập
con, rồi sao chép sang thư mục ghi được — bắt buộc, vì Ultralytics cần ghi tệp `labels.cache`
cạnh thư mục nhãn.

In [5]:
import yaml
from PIL import Image


def find_dataset_root():
    cands = []
    for root in INPUT_ROOTS:
        if not root.exists():
            continue
        for y in root.glob("**/data.yaml"):
            d = y.parent
            score = sum((d / s / "images").is_dir()
                        for s in ("train", "valid", "val", "test"))
            n = len(list((d / "train" / "images").glob("*"))) if (d / "train" / "images").is_dir() else 0
            if score >= 2 and n > 0:
                cands.append((score, n, d))
    if not cands:
        raise FileNotFoundError(
            "Khong tim thay dataset.\n"
            f"Da quet: {[str(r) for r in INPUT_ROOTS]}\n"
            "Kaggle: '+ Add Input' -> pkdarabi/cardetection")
    cands.sort(key=lambda x: (-x[0], -x[1]))
    return cands[0][2]


src_root = find_dataset_root()
if not (DATA / "data.yaml").exists():
    print(f"copy {src_root} -> {DATA} ...")
    shutil.copytree(src_root, DATA, dirs_exist_ok=True)
print(f"dataset: {DATA}")

with open(DATA / "data.yaml") as f:
    _dy = yaml.safe_load(f)
_names = _dy["names"]
CLASSES = [_names[i] for i in sorted(_names)] if isinstance(_names, dict) else list(_names)
NUM_CLASSES = len(CLASSES)
print(f"{NUM_CLASSES} lop: {', '.join(CLASSES[:5])} ...")

IMG_EXT = (".jpg", ".jpeg", ".png", ".bmp", ".webp")
VALID_DIR = "valid" if (DATA / "valid").is_dir() else "val"

copy /kaggle/input/datasets/pkdarabi/cardetection/car -> /kaggle/working/data ...
dataset: /kaggle/working/data
15 lop: Green Light, Red Light, Speed Limit 10, Speed Limit 100, Speed Limit 110 ...


### 1.2 Chuyển nhãn sang COCO cho bộ đánh giá

In [6]:
def split_dir(s):
    return DATA / (VALID_DIR if s == "valid" else s)


def label_path(ip):
    return ip.parent.parent / "labels" / (ip.stem + ".txt")


def read_yolo_label(p):
    if not p.exists():
        return []
    out = []
    for line in p.read_text().strip().splitlines():
        parts = line.split()
        if len(parts) < 5:
            continue
        c = int(float(parts[0]))
        xc, yc, w, h = (float(v) for v in parts[1:5])
        if w > 0 and h > 0:
            out.append((c, xc, yc, w, h))
    return out


def list_images(split):
    d = split_dir(split) / "images"
    imgs = sorted(p for p in d.iterdir() if p.suffix.lower() in IMG_EXT)
    if SMOKE:
        imgs = [p for p in imgs if read_yolo_label(label_path(p))][:SMOKE_N[split]]
    return imgs


def yolo_to_xyxy(raw, W, H):
    """Cat gon CA HAI dau roi loai hop suy bien — neu chi cat mot dau, hop nam
    ngoai khung se cho ra x1 > x2 va lam hong phep do."""
    boxes, labels = [], []
    for c, xc, yc, w, h in raw:
        bw, bh = w * W, h * H
        x1, y1 = xc * W - bw / 2, yc * H - bh / 2
        x2, y2 = x1 + bw, y1 + bh
        x1 = min(max(x1, 0.0), float(W)); x2 = min(max(x2, 0.0), float(W))
        y1 = min(max(y1, 0.0), float(H)); y2 = min(max(y2, 0.0), float(H))
        if x2 - x1 < 1e-3 or y2 - y1 < 1e-3:
            continue
        boxes.append([x1, y1, x2, y2]); labels.append(c + 1)
    return boxes, labels


def yolo_to_coco(split, out_json):
    images, anns, aid = [], [], 1
    for iid, ip in enumerate(list_images(split), start=1):
        with Image.open(ip) as im:
            W, H = im.size
        images.append({"id": iid, "file_name": ip.name, "width": W, "height": H,
                       "abs_path": str(ip)})
        bxs, lbs = yolo_to_xyxy(read_yolo_label(label_path(ip)), W, H)
        for (x1, y1, x2, y2), lb in zip(bxs, lbs):
            bw, bh = x2 - x1, y2 - y1
            anns.append({"id": aid, "image_id": iid, "category_id": lb,
                         "bbox": [round(x1, 2), round(y1, 2), round(bw, 2), round(bh, 2)],
                         "area": round(bw * bh, 2), "iscrowd": 0})
            aid += 1
    Path(out_json).write_text(json.dumps({
        "info": {}, "images": images, "annotations": anns,
        "categories": [{"id": i + 1, "name": c} for i, c in enumerate(CLASSES)]}))
    return len(images), len(anns)


for s in ("train", "valid", "test"):
    ni, na = yolo_to_coco(s, COCO / f"instances_{s}.json")
    print(f"  {s:5s}: {ni:4d} anh, {na:4d} hop")

  train: 3530 anh, 4298 hop
  valid:  801 anh,  944 hop
  test :  638 anh,  770 hop


### 1.3 Kiểm tra nhãn và rò rỉ dữ liệu giữa các tập

> ⚠️ **Bộ dữ liệu này CÓ rò rỉ giữa các tập.** Đo trên toàn bộ 4.969 ảnh: 155 ảnh nguồn xuất
> hiện ở nhiều hơn một tập, khiến **80/638 ảnh kiểm tra (12,5%)** có bản sao ở tập khác, trong
> đó **64 ảnh trùng với tập huấn luyện**. Đây là khiếm khuyết của bộ dữ liệu gốc trên Kaggle.

Không dừng notebook, mà **tách riêng một tập kiểm tra sạch** rồi báo cáo song song hai con số.
Khoảng cách giữa chúng chính là thước đo định lượng cho ảnh hưởng của rò rỉ.

In [7]:
import re

RF_PAT = re.compile(r"^(.*?)\.rf\.[0-9a-f]+\.\w+$")


def source_stem(name):
    """Ten anh nguon truoc khi Roboflow them phan bam."""
    m = RF_PAT.match(name)
    return m.group(1) if m else Path(name).stem


def sha256_of(p, buf=1 << 20):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        while chunk := f.read(buf):
            h.update(chunk)
    return h.hexdigest()


_bad = []
for s in ("train", "valid", "test"):
    d = json.loads((COCO / f"instances_{s}.json").read_text())
    bb = [a for a in d["annotations"] if a["bbox"][2] <= 0 or a["bbox"][3] <= 0]
    bc = [a for a in d["annotations"] if not (1 <= a["category_id"] <= NUM_CLASSES)]
    ok = not (bb or bc) and len(d["images"]) and len(d["annotations"])
    if not ok:
        _bad.append(s)
    print(f"  [{'OK ' if ok else 'LOI'}] {s:5s}: {len(d['images']):4d} anh, "
          f"{len(d['annotations']):4d} hop | suy bien={len(bb)}, sai lop={len(bc)}")
assert not _bad, f"Nhan khong hop le o: {_bad} — day la loi that, phai sua truoc."

_by_stem = {}
for s in ("train", "valid", "test"):
    for ip in list_images(s):
        _by_stem.setdefault(source_stem(ip.name), []).append((s, ip.name))
_cross = {k: v for k, v in _by_stem.items() if len({s for s, _ in v}) > 1}
_train_stems = {source_stem(p.name) for p in list_images("train")}

_tm = json.loads((COCO / "instances_test.json").read_text())
LEAKED_TEST_IDS = {im["id"] for im in _tm["images"]
                   if source_stem(im["file_name"]) in _train_stems}
CLEAN_TEST_IDS = [im["id"] for im in _tm["images"] if im["id"] not in LEAKED_TEST_IDS]
_n_test = len(_tm["images"])

_h = {}
for s in ("train", "valid", "test"):
    for ip in list_images(s):
        _h.setdefault(sha256_of(ip), []).append(s)
_exact = sum(1 for v in _h.values() if len(set(v)) > 1)

print(f"\n{'-'*62}")
print("RO RI GIUA CAC TAP (theo ten anh nguon)")
print(f"{'-'*62}")
print(f"  Nhom anh nguon o >1 tap          : {len(_cross)}")
print(f"  Trong do trung BYTE hoan toan    : {_exact}")
print(f"  Anh KIEM TRA trung voi TAP TRAIN : {len(LEAKED_TEST_IDS)} / {_n_test}"
      f"  ({100*len(LEAKED_TEST_IDS)/max(_n_test,1):.1f}%)")
print(f"  Tap kiem tra SACH con lai        : {len(CLEAN_TEST_IDS)} anh")
if LEAKED_TEST_IDS:
    print("\n!  Ket qua tren TOAN tap kiem tra se bi THOI PHONG.")
    print("!  Bang ket qua se bao cao song song ca hai con so.")

json.dump({"n_cross_groups": len(_cross), "n_exact_byte_dup": _exact,
           "n_test_total": _n_test, "n_test_leaked": len(LEAKED_TEST_IDS),
           "n_test_clean": len(CLEAN_TEST_IDS)},
          open(RESULTS / "metrics" / "data_leakage.json", "w"), indent=2)

  [OK ] train: 3530 anh, 4298 hop | suy bien=0, sai lop=0
  [OK ] valid:  801 anh,  944 hop | suy bien=0, sai lop=0
  [OK ] test :  638 anh,  770 hop | suy bien=0, sai lop=0

--------------------------------------------------------------
RO RI GIUA CAC TAP (theo ten anh nguon)
--------------------------------------------------------------
  Nhom anh nguon o >1 tap          : 155
  Trong do trung BYTE hoan toan    : 101
  Anh KIEM TRA trung voi TAP TRAIN : 65 / 638  (10.2%)
  Tap kiem tra SACH con lai        : 573 anh

!  Ket qua tren TOAN tap kiem tra se bi THOI PHONG.
!  Bang ket qua se bao cao song song ca hai con so.


### 1.4 Hai tệp YAML: một để huấn luyện, một để hiệu chuẩn INT8

> ⚠️ **Bẫy tinh vi:** khi lượng tử hoá INT8, Ultralytics đọc split `val` trong tệp YAML để
> lấy ảnh hiệu chuẩn. Nếu đưa `data.yaml` thường vào, nó sẽ hiệu chuẩn bằng **tập kiểm định**
> — tức rò rỉ dữ liệu. Vì vậy phải tạo YAML riêng, trong đó `val` **trỏ tới ảnh tập huấn luyện**.

In [8]:
DATA_YAML = OUT / "data_train.yaml"
DATA_YAML.write_text(yaml.safe_dump({
    "path": str(DATA), "train": "train/images",
    "val": f"{VALID_DIR}/images", "test": "test/images",
    "nc": NUM_CLASSES, "names": {i: c for i, c in enumerate(CLASSES)},
}, sort_keys=False, allow_unicode=True))

CALIB_DIR = OUT / "calib"
if CALIB_DIR.exists():
    shutil.rmtree(CALIB_DIR)
(CALIB_DIR / "images").mkdir(parents=True)
(CALIB_DIR / "labels").mkdir(parents=True)

_train_imgs = list_images("train")
_by_cls = {}
for ip in _train_imgs:
    for c, *_ in read_yolo_label(label_path(ip)):
        _by_cls.setdefault(c, []).append(ip)

picked, seen = [], set()
_target = min(N_CALIB, len(_train_imgs))
while len(picked) < _target:
    added = False
    for c in sorted(_by_cls):                       # chon phan tang, phu du cac lop
        for ip in _by_cls[c]:
            if ip not in seen:
                picked.append(ip); seen.add(ip); added = True
                break
        if len(picked) >= _target:
            break
    if not added:
        break
for ip in _train_imgs:
    if len(picked) >= _target:
        break
    if ip not in seen:
        picked.append(ip); seen.add(ip)

for ip in picked:
    shutil.copy(ip, CALIB_DIR / "images" / ip.name)
    lp = label_path(ip)
    if lp.exists():
        shutil.copy(lp, CALIB_DIR / "labels" / lp.name)

CALIB_YAML = OUT / "calibration_train.yaml"
CALIB_YAML.write_text(yaml.safe_dump({
    "path": str(CALIB_DIR), "train": "images",
    "val": "images",             # CO Y: anh TAP HUAN LUYEN, khong phai validation
    "nc": NUM_CLASSES, "names": {i: c for i, c in enumerate(CLASSES)},
}, sort_keys=False, allow_unicode=True))

print(f"data_train.yaml       : {DATA_YAML}")
print(f"calibration_train.yaml: {len(picked)} anh (lay tu TAP HUAN LUYEN)")

data_train.yaml       : /kaggle/working/kd_outputs/full/data_train.yaml
calibration_train.yaml: 300 anh (lay tu TAP HUAN LUYEN)


---
## §2 · Huấn luyện ba mô hình

Cấu hình chung cho cả ba, chỉ khác đúng những gì phải khác.

> **Vì sao tắt lật ảnh (`fliplr=0.0`, `flipud=0.0`):** biển "rẽ trái" lật ngang thành biển
> "rẽ phải" — phép tăng cường này dạy mô hình điều sai sự thật. Giữ nhất quán với cấu hình
> của các mô hình đã huấn luyện ở giai đoạn trước.

In [9]:
from ultralytics import YOLO

COMMON = dict(
    data=str(DATA_YAML), imgsz=IMGSZ, seed=SEED, deterministic=True,
    patience=PATIENCE, optimizer="AdamW", lr0=0.001, lrf=0.01, cos_lr=True,
    amp=True, workers=2, device=DEVICE, val=True, plots=True, exist_ok=True,
    save_period=5 if not SMOKE else -1, project=str(RUNS),
    # tang cuong vua phai, KHONG lat anh
    flipud=0.0, fliplr=0.0, degrees=3.0, translate=0.10, scale=0.30,
    shear=0.0, perspective=0.0, hsv_h=0.015, hsv_s=0.4, hsv_v=0.3,
    mosaic=1.0, mixup=0.0, close_mosaic=10 if not SMOKE else 0,
)


def train_stage(stage, model_name, epochs, batch, extra=None, run_name=None):
    """Huan luyen mot giai doan, co the bo qua neu da xong o phien truoc."""
    run_name = run_name or stage.lower()
    best = RUNS / run_name / "weights" / "best.pt"
    if stage_done(stage) and best.exists():
        print(f"[{stage}] da xong o phien truoc -> bo qua  ({best})")
        return best
    if not stage_enabled(stage):
        if best.exists():
            print(f"[{stage}] khong nam trong RUN_STAGE nhung da co weight -> dung lai")
            return best
        raise RuntimeError(f"[{stage}] chua co weight va khong nam trong RUN_STAGE={RUN_STAGE}")

    seed_everything()
    kw = dict(COMMON, epochs=epochs, batch=batch, name=run_name)
    if extra:
        kw.update(extra)
    print(f"\n[{stage}] bat dau — {model_name}, {epochs} epoch, batch {batch}")
    t0 = time.time()
    m = YOLO(model_name)
    m.train(**kw)
    mins = (time.time() - t0) / 60
    print(f"[{stage}] xong trong {mins:.1f} phut -> {best}")
    mark_done(stage, {"stage": stage, "model": model_name, "epochs": epochs,
                      "batch": batch, "minutes": round(mins, 1), "best": str(best)})
    return best

### 2.1 Thầy — `yolo26s`

In [10]:
TEACHER_BEST = train_stage("TEACHER", TEACHER_MODEL, EPOCHS_TEACHER, BATCH_TEACHER)


[TEACHER] bat dau — yolo26s.pt, 50 epoch, batch 16
Ultralytics 8.4.108 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/kaggle/working/kd_outputs/full/data_train.yaml, degrees=3.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.4, hsv_v=0.3, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26s.pt, momentum=0.9

### 2.2 Trò đối chứng — `yolo26n`, huấn luyện thường

Đây là mốc so sánh. Mọi tham số giống hệt trò chưng cất, **trừ** việc không có `distill_model`.

In [11]:
BASELINE_BEST = train_stage("BASELINE", STUDENT_MODEL, EPOCHS_STUDENT, BATCH_BASELINE)


[BASELINE] bat dau — yolo26n.pt, 50 epoch, batch 24
Ultralytics 8.4.108 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=24, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/kaggle/working/kd_outputs/full/data_train.yaml, degrees=3.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.4, hsv_v=0.3, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.pt, momentum=0.

### 2.3 Trò chưng cất — `yolo26n` + `distill_model`

Chỉ thêm đúng hai tham số so với ô trên:

```python
distill_model = <đường dẫn thầy>
dis = 6.0        # trọng số hàm mất mát chưng cất
```

Batch nhỏ hơn vì mô hình thầy cũng phải nằm trong bộ nhớ GPU để suy luận trên từng lô.
Thời gian huấn luyện chậm hơn khoảng 1,2–1,5 lần.

> **Mô hình xuất ra chỉ chứa trọng số của trò.** Khi triển khai, nó có đúng số tham số, đúng
> tốc độ và đúng dung lượng như trò đối chứng — thầy không đi kèm.

In [12]:
KD_BEST = train_stage(
    "KD", STUDENT_MODEL, EPOCHS_STUDENT, BATCH_KD,
    extra={"distill_model": str(TEACHER_BEST), "dis": DIS_WEIGHT},
    run_name="kd")


[KD] bat dau — yolo26n.pt, 50 epoch, batch 8
Ultralytics 8.4.108 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/kaggle/working/kd_outputs/full/data_train.yaml, degrees=3.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=/kaggle/working/kd_outputs/full/runs/teacher/weights/best.pt, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.4, hsv_v=0.3, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixu

### 2.4 Xác minh chưng cất **thật sự** đã được kích hoạt

Truyền tham số vào không có nghĩa là nó chạy. Bằng chứng duy nhất đáng tin là cột `dis_loss`
xuất hiện trong nhật ký huấn luyện. Nếu không có, toàn bộ thí nghiệm vô nghĩa — nên dừng ở đây
thay vì báo cáo một kết quả sai.

In [13]:
import pandas as pd

_kd_csv = RUNS / "kd" / "results.csv"
assert _kd_csv.exists(), f"Khong thay {_kd_csv}"
_kd_df = pd.read_csv(_kd_csv)
_kd_df.columns = [c.strip() for c in _kd_df.columns]
_dis_cols = [c for c in _kd_df.columns if "dis" in c.lower() and "loss" in c.lower()]

print("cac cot trong results.csv cua KD:")
print("   " + ", ".join(_kd_df.columns))

if not _dis_cols:
    raise RuntimeError(
        "\n" + "=" * 66 + "\n"
        "  CHUNG CAT KHONG DUOC KICH HOAT\n" + "=" * 66 + "\n"
        "  Khong tim thay cot 'dis_loss' trong results.csv.\n"
        "  Nghia la tham so distill_model da bi bo qua am tham.\n\n"
        "  Kiem tra:\n"
        "   - duong dan thay co ton tai khong\n"
        "   - thay va tro co cung the he YOLO khong (bat buoc)\n"
        "   - phien ban ultralytics co ho tro chung cat khong\n" + "=" * 66)

_c = _dis_cols[0]
print(f"\nXAC NHAN: chung cat da chay — cot '{_c}'")
print(f"   epoch dau  : {_kd_df[_c].iloc[0]:.4f}")
print(f"   epoch cuoi : {_kd_df[_c].iloc[-1]:.4f}")
print(f"   xu huong   : {'giam (tot)' if _kd_df[_c].iloc[-1] < _kd_df[_c].iloc[0] else 'khong giam'}")

cac cot trong results.csv cua KD:
   epoch, time, train/box_loss, train/cls_loss, train/l1_loss, train/dis_loss, metrics/precision(B), metrics/recall(B), metrics/mAP50(B), metrics/mAP50-95(B), val/box_loss, val/cls_loss, val/l1_loss, val/dis_loss, lr/pg0, lr/pg1, lr/pg2

XAC NHAN: chung cat da chay — cot 'train/dis_loss'
   epoch dau  : 12.1663
   epoch cuoi : 1.1608
   xu huong   : giam (tot)


---
## §3 · Xuất trò chưng cất sang ONNX

> **`end2end=False` là lựa chọn có chủ đích.** YOLO26 mặc định xuất ở chế độ NMS-free, đồ thị
> cho ra tensor `(batch, 300, 6)` với NMS đã nhúng sẵn. Chế độ đó nhanh hơn khi triển khai
> nhưng **không so trực tiếp được** với năm mô hình đã đánh giá ở giai đoạn trước (đều dùng
> hậu xử lý NMS cổ điển). Đặt `end2end=False` giữ nguyên bố cục tensor truyền thống, nhờ đó
> cả bảng kết quả nằm trên cùng một mặt bằng so sánh.

Tham số `quantize=8` là API hiện hành; cờ `int8=True` cũ đã bị đánh dấu lỗi thời. Hàm dưới
thử API mới trước rồi mới lùi về cờ cũ, để chạy được trên cả hai thế hệ thư viện.

In [14]:
def export_model(weights, fmt, quant=None, data=None, tag=""):
    # Ultralytics ghi tep xuat ra NGAY CANH tep .pt nguon. Neu nguon nam o thu muc
    # chi doc (vi du /kaggle/input) thi export se that bai voi "Read-only file system".
    weights = Path(weights)
    if not os.access(weights.parent, os.W_OK):
        local = EXPORTS / f"src_{weights.name}"
        if not local.exists():
            shutil.copy(weights, local)
        print(f"   (nguon chi doc -> dung ban sao {local.name})")
        weights = local
    m = YOLO(str(weights))
    kw = dict(format=fmt, imgsz=IMGSZ, batch=1, dynamic=False, device="cpu",
              end2end=False)
    if fmt == "onnx":
        kw.update(opset=19, simplify=True)
    if data:
        kw["data"] = str(data)
    trials = ({16: [{"quantize": 16}, {"half": True}],
               8: [{"quantize": 8}, {"int8": True}]}.get(quant, [{}]))
    last = None
    for extra in trials:
        try:
            out = m.export(**kw, **extra)
            print(f"   [OK] {tag or fmt}  ({list(extra) or 'FP32'})")
            return Path(out)
        except TypeError as e:
            last = e; continue
        except Exception as e:
            last = e; break
    raise RuntimeError(f"Xuat {tag or fmt} that bai: {last}")


print("[1/2] ONNX FP32 ...")
_p = export_model(KD_BEST, "onnx", None, tag="ONNX FP32")
KD_ONNX_FP32 = EXPORTS / "student_kd_fp32.onnx"
shutil.copy(_p, KD_ONNX_FP32)          # copy ngay, lan xuat sau ghi de cung ten

print("[2/2] ONNX INT8 (hieu chuan tren tap train) ...")
_p = export_model(KD_BEST, "onnx", 8, CALIB_YAML, tag="ONNX INT8")
KD_ONNX_INT8 = EXPORTS / "student_kd_int8.onnx"
shutil.copy(_p, KD_ONNX_INT8)

for f in sorted(EXPORTS.iterdir()):
    print(f"   {f.name:30s} {f.stat().st_size/1024**2:7.2f} MB")

[1/2] ONNX FP32 ...
Ultralytics 8.4.108 🚀 Python-3.12.13 torch-2.10.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/
YOLO26n summary (fused): 146 layers, 2,500,154 parameters, 0 gradients, 5.2 GFLOPs

PyTorch: starting from '/kaggle/working/kd_outputs/full/runs/kd/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 19, 8400) (5.1 MB)

ONNX: starting export with onnx 1.22.0 opset 19...
ONNX: slimming with onnxslim 0.1.94...
ONNX: export success ✅ 2.0s, saved as '/kaggle/working/kd_outputs/full/runs/kd/weights/best.onnx' (9.4 MB)

Export complete (2.4s)
Results saved to /kaggle/working/kd_outputs/full/runs/kd/weights/best.onnx
Predict:         yolo predict task=detect model=/kaggle/working/kd_outputs/full/runs/kd/weights/best.onnx imgsz=640 
Validate:        yolo val task=detect model=/kaggle/working/kd_outputs/full/runs/kd/weight

ONNX: export success ✅ 59.9s, saved as '/kaggle/working/kd_outputs/full/runs/kd/weights/best_int8.onnx' (2.8 MB)

Export complete (60.3s)
Results saved to /kaggle/working/kd_outputs/full/runs/kd/weights/best_int8.onnx
Predict:         yolo predict task=detect model=/kaggle/working/kd_outputs/full/runs/kd/weights/best_int8.onnx imgsz=640 
Validate:        yolo val task=detect model=/kaggle/working/kd_outputs/full/runs/kd/weights/best_int8.onnx imgsz=640 data=/kaggle/working/kd_outputs/full/data_train.yaml  
Visualize:       https://netron.app
   [OK] ONNX INT8  (['quantize'])
   student_kd_fp32.onnx              9.35 MB
   student_kd_int8.onnx              2.78 MB


### 3.1 Kiểm chứng tệp đã xuất

Xuất xong không có nghĩa là dùng được. Kiểm tra bốn thứ: đồ thị hợp lệ, nạp được, chạy được
một ảnh, kết quả không chứa NaN.

In [15]:
import onnx
import onnxruntime as ort

_chk = {}
for nm, pth in [("ONNX FP32", KD_ONNX_FP32), ("ONNX INT8", KD_ONNX_INT8)]:
    rec = {"path": str(pth), "size_mb": round(pth.stat().st_size / 1024**2, 2)}
    try:
        mo = onnx.load(str(pth)); onnx.checker.check_model(mo)
        ops = [n.op_type for n in mo.graph.node]
        rec.update(graph_valid=True, n_nodes=len(ops),
                   n_quant_nodes=sum(o in ("QuantizeLinear", "DequantizeLinear",
                                           "QLinearConv", "QLinearMatMul") for o in ops))
        rec["mixed_precision"] = 0 < rec["n_quant_nodes"] < len(ops)
        sess = ort.InferenceSession(str(pth), providers=["CPUExecutionProvider"])
        inp = sess.get_inputs()[0]
        y = sess.run(None, {inp.name: np.zeros([1, 3, IMGSZ, IMGSZ], dtype=np.float32)})
        rec["output_shapes"] = [list(o.shape) for o in y]
        rec["has_nan"] = bool(any(not np.isfinite(o).all() for o in y))
        rec["runtime_ok"] = True
    except Exception as e:
        rec.update(runtime_ok=False, error=str(e)[:300])
    _chk[nm] = rec
    print(f"{nm}: {rec.get('size_mb')} MB | graph={rec.get('graph_valid')} | "
          f"runtime={rec.get('runtime_ok')} | nut luong tu hoa={rec.get('n_quant_nodes')} | "
          f"mixed={rec.get('mixed_precision')} | NaN={rec.get('has_nan')}")

(RESULTS / "metrics" / "onnx_check.json").write_text(json.dumps(_chk, indent=2))
for k in _chk:
    assert _chk[k]["runtime_ok"] and not _chk[k].get("has_nan", True), f"{k} khong dung duoc"
print("\nCa hai tep ONNX deu nap va suy luan duoc.")

ONNX FP32: 9.35 MB | graph=True | runtime=True | nut luong tu hoa=0 | mixed=False | NaN=False
ONNX INT8: 2.78 MB | graph=True | runtime=True | nut luong tu hoa=616 | mixed=True | NaN=False

Ca hai tep ONNX deu nap va suy luan duoc.


---
## §4 · Đánh giá năm cấu hình bằng một bộ đo duy nhất

Cả năm đều nạp qua `YOLO(...)`, nên **đường tiền xử lý và hậu xử lý hoàn toàn giống nhau**.
Khác biệt duy nhất là trọng số và định dạng. Đo bằng `pycocotools` trên cùng tập kiểm tra —
cùng một bộ đo đã dùng cho năm mô hình ở giai đoạn trước, nên kết quả ghép chung bảng được.

In [16]:
from pycocotools.coco import COCO as PyCOCO
from pycocotools.cocoeval import COCOeval
import contextlib, io

TEST_META = json.loads((COCO / "instances_test.json").read_text())
TEST_IMAGES = TEST_META["images"]
print(f"tap kiem tra: {len(TEST_IMAGES)} anh, {len(TEST_META['annotations'])} hop")


class Adapter:
    """Giao dien chung: anh -> (hop xyxy, diem tin cay, nhan 1-based)."""

    def __init__(self, name, path, device, fmt, precision, role):
        self.name, self.path = name, Path(path)
        self.device, self.fmt = device, fmt
        self.precision, self.role = precision, role
        self.m = YOLO(str(path), task="detect")

    def predict(self, pil):
        r = self.m.predict(pil, conf=CONF_EVAL, iou=0.7, max_det=300,
                           imgsz=IMGSZ, device=self.device, verbose=False)[0]
        b = r.boxes
        if b is None or len(b) == 0:
            return np.zeros((0, 4)), np.zeros(0), np.zeros(0, dtype=int)
        return (b.xyxy.cpu().numpy(), b.conf.cpu().numpy(),
                b.cls.cpu().numpy().astype(int) + 1)

    def n_params(self):
        try:
            return sum(p.numel() for p in self.m.model.parameters())
        except Exception:
            return float("nan")

    def size_mb(self):
        if self.path.is_dir():
            return sum(f.stat().st_size for f in self.path.rglob("*") if f.is_file()) / 1024**2
        return self.path.stat().st_size / 1024**2


def coco_eval(gt_dict, dets, img_ids=None):
    """img_ids=None -> toan tap; truyen danh sach -> chi cac anh do."""
    gt = {k: v for k, v in gt_dict.items()}
    gt["images"] = [{k: v for k, v in im.items() if k != "abs_path"} for im in gt_dict["images"]]
    if not dets:
        return None, None
    with contextlib.redirect_stdout(io.StringIO()):
        c = PyCOCO(); c.dataset = gt; c.createIndex()
        e = COCOeval(c, c.loadRes(list(dets)), "bbox")
        if img_ids is not None:
            e.params.imgIds = sorted(img_ids)
        e.evaluate(); e.accumulate(); e.summarize()
    s = e.stats
    return {"map50_95": float(s[0]), "map50": float(s[1]), "map75": float(s[2]),
            "map_small": float(s[3]), "map_medium": float(s[4]), "map_large": float(s[5]),
            "recall_100": float(s[8])}, e


def per_class_ap(e, names):
    prec = e.eval["precision"]
    out = {}
    for k, nm in enumerate(names):
        p = prec[:, :, k, 0, 2]; p = p[p > -1]
        out[nm] = float(np.mean(p)) if p.size else float("nan")
    return out


def class_agnostic_eval(gt_dict, dets):
    """Gop 15 lop thanh 1 -> chi do kha nang DONG KHUNG."""
    gt = json.loads(json.dumps({k: v for k, v in gt_dict.items() if k != "images"}))
    gt["images"] = [{k: v for k, v in im.items() if k != "abs_path"} for im in gt_dict["images"]]
    for a in gt["annotations"]:
        a["category_id"] = 1
    gt["categories"] = [{"id": 1, "name": "sign"}]
    m, _ = coco_eval(gt, [dict(d, category_id=1) for d in dets])
    return m


def iou_matrix(a, b):
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)))
    ax1, ay1, ax2, ay2 = a[:, 0:1], a[:, 1:2], a[:, 2:3], a[:, 3:4]
    bx1, by1, bx2, by2 = b[:, 0], b[:, 1], b[:, 2], b[:, 3]
    iw = np.clip(np.minimum(ax2, bx2) - np.maximum(ax1, bx1), 0, None)
    ih = np.clip(np.minimum(ay2, by2) - np.maximum(ay1, by1), 0, None)
    inter = iw * ih
    ua = (ax2 - ax1) * (ay2 - ay1) + (bx2 - bx1) * (by2 - by1) - inter
    return inter / np.maximum(ua, 1e-9)


def build_confusion(cache, gt_dict, conf=CONF_VIZ, iou_thr=IOU_MATCH):
    K = NUM_CLASSES
    cm = np.zeros((K + 1, K + 1), dtype=int)
    gt_by = {}
    for a in gt_dict["annotations"]:
        x, y, w, h = a["bbox"]
        gt_by.setdefault(a["image_id"], []).append(([x, y, x + w, y + h], a["category_id"] - 1))
    for im in gt_dict["images"]:
        gts = gt_by.get(im["id"], [])
        gb = np.array([g[0] for g in gts], float).reshape(-1, 4)
        gc = np.array([g[1] for g in gts], int)
        boxes, scores, labels = cache.get(im["id"],
                                          (np.zeros((0, 4)), np.zeros(0), np.zeros(0, int)))
        keep = scores >= conf
        boxes, scores, labels = boxes[keep], scores[keep], labels[keep] - 1
        labels = np.clip(labels, 0, K - 1)
        o = np.argsort(-scores); boxes, labels = boxes[o], labels[o]
        ious, used = iou_matrix(boxes, gb), set()
        for pi in range(len(boxes)):
            gi, best = -1, iou_thr
            for gj in range(len(gb)):
                if gj not in used and ious[pi, gj] >= best:
                    best, gi = ious[pi, gj], gj
            if gi >= 0:
                used.add(gi); cm[gc[gi], labels[pi]] += 1
            else:
                cm[K, labels[pi]] += 1
        for gj in range(len(gb)):
            if gj not in used:
                cm[gc[gj], K] += 1
    return cm


def measure_speed(ad, warmup=N_SPEED_WARMUP, runs=N_SPEED_RUNS):
    pool = [Image.open(im["abs_path"]).convert("RGB")
            for im in TEST_IMAGES[:max(warmup, runs)]]
    for i in range(warmup):
        ad.predict(pool[i % len(pool)])
    if ad.device != "cpu":
        torch.cuda.synchronize()
    ts = []
    for i in range(runs):
        t0 = time.perf_counter()
        ad.predict(pool[i % len(pool)])
        if ad.device != "cpu":
            torch.cuda.synchronize()
        ts.append((time.perf_counter() - t0) * 1000)
    for p in pool:
        p.close()
    ts = np.array(ts); med = float(np.median(ts))
    return {"latency_mean_ms": round(float(ts.mean()), 2),
            "latency_median_ms": round(med, 2),
            "latency_p95_ms": round(float(np.percentile(ts, 95)), 2),
            "fps": round(1000 / med, 1), "warmup_runs": warmup, "timed_runs": runs}


def run_inference(ad, images=TEST_IMAGES):
    dets, cache, t0 = [], {}, time.time()
    for i, im in enumerate(images, 1):
        with Image.open(im["abs_path"]) as pil:
            pil = pil.convert("RGB")
            b, s, l = ad.predict(pil)
        cache[im["id"]] = (b, s, l)
        for (x1, y1, x2, y2), sc, c in zip(b, s, l):
            dets.append({"image_id": im["id"], "category_id": int(c),
                         "bbox": [float(x1), float(y1), float(x2 - x1), float(y2 - y1)],
                         "score": float(sc)})
        if i % 200 == 0 or i == len(images):
            print(f"   [{ad.name}] {i}/{len(images)} ({time.time()-t0:.0f}s)")
    return dets, cache


def evaluate(ad):
    print(f"\n{'='*64}\n  {ad.name}\n{'='*64}")
    dets, cache = run_inference(ad)
    overall, e = coco_eval(TEST_META, dets)
    if overall is None:
        print("  !! khong co du doan nao"); return None
    pc = per_class_ap(e, CLASSES)
    agn = class_agnostic_eval(TEST_META, dets)
    cm = build_confusion(cache, TEST_META)
    spd = measure_speed(ad)
    matched = cm[:NUM_CLASSES, :NUM_CLASSES]
    cls_acc = float(np.trace(matched) / max(matched.sum(), 1))
    npar = ad.n_params()
    clean = {}
    if LEAKED_TEST_IDS and len(CLEAN_TEST_IDS) >= 10:
        co, _ = coco_eval(TEST_META, dets, img_ids=CLEAN_TEST_IDS)
        if co:
            clean = {f"clean_{k}": v for k, v in co.items()}
    m = {"model": ad.name, "role": ad.role, "format": ad.fmt, "precision": ad.precision,
         "device": str(ad.device), **overall, **clean,
         "n_test_all": len(TEST_IMAGES), "n_test_clean": len(CLEAN_TEST_IDS),
         "localization_only_map50": agn["map50"],
         "localization_only_recall": agn["recall_100"],
         "classification_accuracy_given_match": round(cls_acc, 4),
         "params": npar, "params_m": round(npar / 1e6, 2) if npar == npar else None,
         "size_mb": round(ad.size_mb(), 2), "imgsz": IMGSZ, "batch_size": 1,
         **spd, "per_class_ap": pc, "confusion_matrix": cm.tolist()}
    (RESULTS / "metrics" / f"{ad.name.replace(' ', '_').replace('/', '_').lower()}.json"
     ).write_text(json.dumps(m, indent=2))
    print(f"  mAP@0.5        : {m['map50']:.4f}")
    print(f"  mAP@0.5:0.95   : {m['map50_95']:.4f}")
    if clean:
        print(f"  mAP@0.5 (SACH) : {m['clean_map50']:.4f}   "
              f"({m['clean_map50']-m['map50']:+.4f} so voi toan tap, "
              f"tren {len(CLEAN_TEST_IDS)} anh)")
    print(f"  AP nho/vua/lon : {m['map_small']:.3f} / {m['map_medium']:.3f} / {m['map_large']:.3f}")
    print(f"  chi DINH VI    : {m['localization_only_map50']:.4f}")
    print(f"  chi PHAN LOAI  : {cls_acc:.4f}")
    print(f"  do tre trung vi: {m['latency_median_ms']} ms (p95 {m['latency_p95_ms']})")
    print(f"  kich thuoc     : {m['params_m']} M | {m['size_mb']} MB  [{ad.device}]")
    return m


CONFIGS = [
    ("Thay yolo26s",        TEACHER_BEST,  DEVICE, "pytorch", "FP16", "teacher"),
    ("Tro doi chung",       BASELINE_BEST, DEVICE, "pytorch", "FP16", "student_baseline"),
    ("Tro chung cat",       KD_BEST,       DEVICE, "pytorch", "FP16", "student_kd"),
    ("Tro chung cat ONNX FP32", KD_ONNX_FP32, "cpu", "onnx", "FP32", "student_kd_onnx"),
    ("Tro chung cat ONNX INT8", KD_ONNX_INT8, "cpu", "onnx", "INT8", "student_kd_onnx"),
]

all_metrics = []
for nm, pth, dev, fmt, prec, role in CONFIGS:
    try:
        all_metrics.append(evaluate(Adapter(nm, pth, dev, fmt, prec, role)))
    except Exception as ex:
        print(f"\n!! BO QUA {nm}: {ex}")
all_metrics = [m for m in all_metrics if m]
print(f"\nDa danh gia {len(all_metrics)}/{len(CONFIGS)} cau hinh.")

tap kiem tra: 638 anh, 770 hop

  Thay yolo26s
   [Thay yolo26s] 200/638 (3s)
   [Thay yolo26s] 400/638 (6s)
   [Thay yolo26s] 600/638 (9s)
   [Thay yolo26s] 638/638 (10s)
  mAP@0.5        : 0.9520
  mAP@0.5:0.95   : 0.7822
  mAP@0.5 (SACH) : 0.9643   (+0.0123 so voi toan tap, tren 573 anh)
  AP nho/vua/lon : 0.616 / 0.814 / 0.859
  chi DINH VI    : 0.9654
  chi PHAN LOAI  : 0.9917
  do tre trung vi: 13.44 ms (p95 14.15)
  kich thuoc     : 9.47 M | 19.39 MB  [0]

  Tro doi chung
   [Tro doi chung] 200/638 (3s)
   [Tro doi chung] 400/638 (6s)
   [Tro doi chung] 600/638 (9s)
   [Tro doi chung] 638/638 (9s)
  mAP@0.5        : 0.9348
  mAP@0.5:0.95   : 0.7773
  mAP@0.5 (SACH) : 0.9498   (+0.0150 so voi toan tap, tren 573 anh)
  AP nho/vua/lon : 0.548 / 0.798 / 0.862
  chi DINH VI    : 0.9517
  chi PHAN LOAI  : 0.9829
  do tre trung vi: 14.35 ms (p95 14.98)
  kich thuoc     : 2.38 M | 5.15 MB  [0]

  Tro chung cat
   [Tro chung cat] 200/638 (3s)
   [Tro chung cat] 400/638 (6s)
   [Tro chung

---
## §5 · Bảng kết quả

In [17]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

pd.set_option("display.width", 240); pd.set_option("display.max_columns", 40)

df = pd.DataFrame([{
    "Cấu hình": m["model"], "Vai trò": m["role"], "Định dạng": m["format"],
    "Số học": m["precision"], "Thiết bị": m["device"],
    "Tham số (M)": m["params_m"], "Tệp (MB)": m["size_mb"],
    "mAP@0.5": round(m["map50"], 4), "mAP@.5:.95": round(m["map50_95"], 4),
    "AP nhỏ": round(m["map_small"], 3), "AP lớn": round(m["map_large"], 3),
    "Chỉ định vị": round(m["localization_only_map50"], 4),
    "Chỉ phân loại": m["classification_accuracy_given_match"],
    "Trung vị (ms)": m["latency_median_ms"], "p95 (ms)": m["latency_p95_ms"],
    "FPS": m["fps"],
} for m in all_metrics])
df.to_csv(RESULTS / "tables" / "kd_comparison.csv", index=False)
print(df.to_string(index=False))
df

               Cấu hình          Vai trò Định dạng Số học Thiết bị  Tham số (M)  Tệp (MB)  mAP@0.5  mAP@.5:.95  AP nhỏ  AP lớn  Chỉ định vị  Chỉ phân loại  Trung vị (ms)  p95 (ms)  FPS
           Thay yolo26s          teacher   pytorch   FP16        0         9.47     19.39   0.9520      0.7822   0.616   0.859       0.9654         0.9917          13.44     14.15 74.4
          Tro doi chung student_baseline   pytorch   FP16        0         2.38      5.15   0.9348      0.7773   0.548   0.862       0.9517         0.9829          14.35     14.98 69.7
          Tro chung cat       student_kd   pytorch   FP16        0         2.38      5.15   0.9014      0.7450   0.486   0.852       0.9403         0.9753          13.37     14.33 74.8
Tro chung cat ONNX FP32  student_kd_onnx      onnx   FP32      cpu          NaN      9.35   0.9079      0.7473   0.462   0.847       0.9595         0.9653          50.41     55.76 19.8
Tro chung cat ONNX INT8  student_kd_onnx      onnx   INT8      cpu         

,Cấu hình,Vai trò,Định dạng,Số học,Thiết bị,Tham số (M),Tệp (MB),mAP@0.5,mAP@.5:.95,AP nhỏ,AP lớn,Chỉ định vị,Chỉ phân loại,Trung vị (ms),p95 (ms),FPS
0,Thay yolo26s,teacher,pytorch,FP16,0,9.47,19.39,0.9520,0.7822,0.616,0.859,0.9654,0.9917,13.44,14.15,74.4
1,Tro doi chung,student_baseline,pytorch,FP16,0,2.38,5.15,0.9348,0.7773,0.548,0.862,0.9517,0.9829,14.35,14.98,69.7
2,Tro chung cat,student_kd,pytorch,FP16,0,2.38,5.15,0.9014,0.7450,0.486,0.852,0.9403,0.9753,13.37,14.33,74.8
3,Tro chung cat ONNX FP32,student_kd_onnx,onnx,FP32,cpu,NaN,9.35,0.9079,0.7473,0.462,0.847,0.9595,0.9653,50.41,55.76,19.8
4,Tro chung cat ONNX INT8,student_kd_onnx,onnx,INT8,cpu,NaN,2.78,0.9041,0.7437,0.437,0.839,0.9523,0.9577,106.90,112.88,9.4


### 5.1 Ba phép so sánh cốt lõi

Bảng trên chỉ là số liệu thô. Ba phép so sánh dưới đây mới trả lời câu hỏi nghiên cứu.

In [18]:
def get(role, prec=None):
    for m in all_metrics:
        if m["role"] == role and (prec is None or m["precision"] == prec):
            return m
    return None


T, B, K = get("teacher"), get("student_baseline"), get("student_kd")
KF = get("student_kd_onnx", "FP32")
KI = get("student_kd_onnx", "INT8")

comps = []
if B and K:
    comps.append({
        "Phép so sánh": "Chưng cất có giúp gì không?",
        "Đối tượng": "Trò chưng cất vs Trò đối chứng",
        "Δ mAP@0.5": round(K["map50"] - B["map50"], 4),
        "Δ mAP@.5:.95": round(K["map50_95"] - B["map50_95"], 4),
        "Δ AP nhỏ": round(K["map_small"] - B["map_small"], 4),
        "Δ Tệp (MB)": round(K["size_mb"] - B["size_mb"], 2),
        "Δ Độ trễ (ms)": round(K["latency_median_ms"] - B["latency_median_ms"], 2),
    })
if T and K:
    comps.append({
        "Phép so sánh": "Trò thu hẹp được bao nhiêu so với thầy?",
        "Đối tượng": "Trò chưng cất vs Thầy",
        "Δ mAP@0.5": round(K["map50"] - T["map50"], 4),
        "Δ mAP@.5:.95": round(K["map50_95"] - T["map50_95"], 4),
        "Δ AP nhỏ": round(K["map_small"] - T["map_small"], 4),
        "Δ Tệp (MB)": round(K["size_mb"] - T["size_mb"], 2),
        "Δ Độ trễ (ms)": round(K["latency_median_ms"] - T["latency_median_ms"], 2),
    })
if KF and KI:
    comps.append({
        "Phép so sánh": "Lượng tử hoá INT8 lấy đi bao nhiêu?",
        "Đối tượng": "ONNX INT8 vs ONNX FP32",
        "Δ mAP@0.5": round(KI["map50"] - KF["map50"], 4),
        "Δ mAP@.5:.95": round(KI["map50_95"] - KF["map50_95"], 4),
        "Δ AP nhỏ": round(KI["map_small"] - KF["map_small"], 4),
        "Δ Tệp (MB)": round(KI["size_mb"] - KF["size_mb"], 2),
        "Δ Độ trễ (ms)": round(KI["latency_median_ms"] - KF["latency_median_ms"], 2),
    })

df_cmp = pd.DataFrame(comps)
df_cmp.to_csv(RESULTS / "tables" / "key_comparisons.csv", index=False)
print(df_cmp.to_string(index=False))
print("\nSo duong = tot hon o cot do chinh xac; so am = nho hon/nhanh hon o cot tep/do tre.")
df_cmp

                           Phép so sánh                      Đối tượng  Δ mAP@0.5  Δ mAP@.5:.95  Δ AP nhỏ  Δ Tệp (MB)  Δ Độ trễ (ms)
            Chưng cất có giúp gì không? Trò chưng cất vs Trò đối chứng    -0.0334       -0.0323   -0.0620        0.00          -0.98
Trò thu hẹp được bao nhiêu so với thầy?          Trò chưng cất vs Thầy    -0.0507       -0.0372   -0.1308      -14.24          -0.07
    Lượng tử hoá INT8 lấy đi bao nhiêu?         ONNX INT8 vs ONNX FP32    -0.0038       -0.0036   -0.0247       -6.57          56.49

So duong = tot hon o cot do chinh xac; so am = nho hon/nhanh hon o cot tep/do tre.


,Phép so sánh,Đối tượng,Δ mAP@0.5,Δ mAP@.5:.95,Δ AP nhỏ,Δ Tệp (MB),Δ Độ trễ (ms)
0,Chưng cất có giúp gì không?,Trò chưng cất vs Trò đối chứng,-0.0334,-0.0323,-0.0620,0.00,-0.98
1,Trò thu hẹp được bao nhiêu so với thầy?,Trò chưng cất vs Thầy,-0.0507,-0.0372,-0.1308,-14.24,-0.07
2,Lượng tử hoá INT8 lấy đi bao nhiêu?,ONNX INT8 vs ONNX FP32,-0.0038,-0.0036,-0.0247,-6.57,56.49


### 5.2 Kiểm tra các ngưỡng chấp nhận

Đặt ngưỡng **trước** khi nhìn kết quả là cách tránh tự biện minh cho số liệu bất lợi. Ngưỡng
đã khai báo ở ô cấu hình §0.

> Không đạt ngưỡng **không phải thất bại**. Kết quả âm vẫn báo cáo được, miễn là trung thực và
> có giải thích. Điều không chấp nhận được là điều chỉnh ngưỡng sau khi đã thấy kết quả.

In [19]:
checks = []
if B and K:
    d = K["map50_95"] - B["map50_95"]
    checks.append(("KD không kém trò đối chứng quá 0,5 điểm",
                   f"Δ = {d:+.4f}", d >= THRESH["kd_not_worse_than_baseline"]))
    checks.append(("KD tốt hơn trò đối chứng (kỳ vọng)",
                   f"Δ = {d:+.4f}", d > 0))
if KF and KI:
    d = KF["map50_95"] - KI["map50_95"]
    checks.append(("INT8 giảm không quá 1,5 điểm so với FP32",
                   f"giảm {d:.4f}", d <= THRESH["int8_max_drop"]))
    r = 1 - KI["size_mb"] / KF["size_mb"]
    checks.append((f"INT8 nhỏ hơn FP32 ít nhất {THRESH['int8_min_size_reduction']:.0%}",
                   f"giảm {r:.1%}", r >= THRESH["int8_min_size_reduction"]))
if K:
    checks.append(("Trò chưng cất nhẹ hơn thầy",
                   f"{K['size_mb']} vs {T['size_mb']} MB" if T else "-",
                   bool(T and K["size_mb"] < T["size_mb"])))

print(f"{'Ngưỡng':<52}{'Giá trị đo':<22}{'Kết quả'}")
print("-" * 88)
for name, val, ok in checks:
    print(f"{name:<52}{val:<22}{'DAT' if ok else 'KHONG DAT'}")
pd.DataFrame([{"tieu_chi": n, "gia_tri": v, "dat": bool(o)} for n, v, o in checks]
             ).to_csv(RESULTS / "tables" / "threshold_checks.csv", index=False)
print(f"\n{sum(o for _,_,o in checks)}/{len(checks)} nguong dat.")

Ngưỡng                                              Giá trị đo            Kết quả
----------------------------------------------------------------------------------------
KD không kém trò đối chứng quá 0,5 điểm             Δ = -0.0323           KHONG DAT
KD tốt hơn trò đối chứng (kỳ vọng)                  Δ = -0.0323           KHONG DAT
INT8 giảm không quá 1,5 điểm so với FP32            giảm 0.0036           DAT
INT8 nhỏ hơn FP32 ít nhất 50%                       giảm 70.3%            DAT
Trò chưng cất nhẹ hơn thầy                          5.15 vs 19.39 MB      DAT

3/5 nguong dat.


### 5.3 Biểu đồ

In [20]:
if len(df):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    x = np.arange(len(df))
    cols = ["#1F3864" if r == "teacher" else "#2E5C9A" if "baseline" in r
            else "#D9821B" for r in df["Vai trò"]]
    axes[0].bar(x, df["mAP@0.5"], color=cols, edgecolor="black", linewidth=0.6)
    axes[0].set_xticks(x); axes[0].set_xticklabels(df["Cấu hình"], rotation=22, ha="right")
    axes[0].set_ylabel("mAP@0.5"); axes[0].set_title("Do chinh xac")
    axes[0].grid(axis="y", alpha=0.3)
    for i, v in enumerate(df["mAP@0.5"]):
        axes[0].text(i, v + 0.006, f"{v:.3f}", ha="center", fontsize=8)

    axes[1].bar(x, df["Tệp (MB)"], color=cols, edgecolor="black", linewidth=0.6)
    axes[1].set_xticks(x); axes[1].set_xticklabels(df["Cấu hình"], rotation=22, ha="right")
    axes[1].set_ylabel("MB"); axes[1].set_title("Dung luong tep")
    axes[1].grid(axis="y", alpha=0.3)
    for i, v in enumerate(df["Tệp (MB)"]):
        axes[1].text(i, v + 0.2, f"{v:.1f}", ha="center", fontsize=8)
    fig.tight_layout(); fig.savefig(RESULTS / "plots" / "accuracy_and_size.png", dpi=150)
    plt.show()

In [21]:
if B and K:
    fig, ax = plt.subplots(figsize=(9, 5))
    labels = ["mAP@0.5", "mAP@.5:.95", "AP nhỏ", "AP vừa", "AP lớn"]
    keys = ["map50", "map50_95", "map_small", "map_medium", "map_large"]
    x = np.arange(len(labels)); w = 0.35
    ax.bar(x - w/2, [B[k] for k in keys], w, label="Tro doi chung",
           color="#2E5C9A", edgecolor="black", linewidth=0.6)
    ax.bar(x + w/2, [K[k] for k in keys], w, label="Tro chung cat",
           color="#D9821B", edgecolor="black", linewidth=0.6)
    ax.set_xticks(x); ax.set_xticklabels(labels)
    ax.set_ylabel("diem"); ax.legend(); ax.grid(axis="y", alpha=0.3)
    ax.set_title("Chung cat giup o dau? — cung kien truc, cung ngan sach huan luyen")
    for i, k in enumerate(keys):
        d = K[k] - B[k]
        ax.text(i, max(B[k], K[k]) + 0.02, f"{d:+.3f}",
                ha="center", fontsize=8,
                color=("green" if d > 0 else "red"), weight="bold")
    fig.tight_layout()
    fig.savefig(RESULTS / "plots" / "kd_vs_baseline.png", dpi=150)
    plt.show()

In [22]:
if _dis_cols:
    fig, ax = plt.subplots(figsize=(9, 4.5))
    ax.plot(_kd_df["epoch"] if "epoch" in _kd_df.columns else range(len(_kd_df)),
            _kd_df[_dis_cols[0]], lw=1.8, color="#D9821B", label=_dis_cols[0])
    ax.set_xlabel("epoch"); ax.set_ylabel("dis_loss")
    ax.set_title("Ham mat mat chung cat — bang chung KD that su duoc kich hoat")
    ax.grid(alpha=0.3); ax.legend()
    fig.tight_layout(); fig.savefig(RESULTS / "plots" / "distillation_loss.png", dpi=150)
    plt.show()

In [23]:
if all_metrics:
    pc = pd.DataFrame({m["model"]: m["per_class_ap"] for m in all_metrics})
    pc.to_csv(RESULTS / "tables" / "per_class_ap.csv")
    if B and K:
        bn, kn = B["model"], K["model"]
        delta = (pc[kn] - pc[bn]).sort_values()
        print("Lop duoc chung cat giup NHIEU nhat:")
        for k, v in delta.tail(5)[::-1].items():
            print(f"   {k:<20} {v:+.4f}")
        print("\nLop bi anh huong XAU nhat:")
        for k, v in delta.head(3).items():
            print(f"   {k:<20} {v:+.4f}")
    pc.round(3)

Lop duoc chung cat giup NHIEU nhat:
   Green Light          +0.0122
   Speed Limit 40       +0.0053
   Speed Limit 110      +0.0011
   Speed Limit 80       -0.0091
   Speed Limit 60       -0.0108

Lop bi anh huong XAU nhat:
   Speed Limit 10       -0.1964
   Red Light            -0.0557
   Speed Limit 70       -0.0473


---
## §6 · Báo cáo và đóng gói

In [24]:
lines = ["# Kết quả chưng cất tri thức — YOLO26s dạy YOLO26n", ""]
lines += [f"- Chế độ chạy: **{RUN_MODE}**",
          f"- GPU: {torch.cuda.get_device_name(0)}",
          f"- Ultralytics: {ultralytics.__version__}",
          f"- Thầy: `{TEACHER_MODEL}` · Trò: `{STUDENT_MODEL}` · `dis={DIS_WEIGHT}`",
          f"- Số epoch: thầy {EPOCHS_TEACHER}, trò {EPOCHS_STUDENT} (cả hai trò như nhau)",
          f"- Ảnh vào: {IMGSZ} · Hạt giống: {SEED}",
          f"- Tập kiểm tra: {len(TEST_IMAGES)} ảnh · Bộ đo: pycocotools", ""]
lines += ["## Bảng kết quả", "", df.to_markdown(index=False), ""]
if len(df_cmp):
    lines += ["## Ba phép so sánh cốt lõi", "", df_cmp.to_markdown(index=False), ""]
lines += ["## Kiểm tra ngưỡng", "",
          "| Tiêu chí | Giá trị đo | Kết quả |", "|---|---|---|"]
lines += [f"| {n} | {v} | {'ĐẠT' if o else 'KHÔNG ĐẠT'} |" for n, v, o in checks]
lines += ["", "## Hạn chế", "",
          "- Số đo tốc độ lấy trên phần cứng Kaggle/Colab, **không đại diện cho thiết bị biên**.",
          "  Muốn kết luận về tiết kiệm năng lượng phải đo lại trên đúng phần cứng đích.",
          "- Dòng chạy trên GPU và dòng chạy trên CPU **không so trực tiếp với nhau được**.",
          "- Mô hình ONNX xuất với `end2end=False` để so sánh được với các mô hình ở giai đoạn",
          "  trước; chế độ NMS-free mặc định của YOLO26 có thể cho tốc độ khác.",
          "- Lớp hiếm (Speed Limit 10) có quá ít mẫu trong tập kiểm tra, mọi chỉ số riêng của nó",
          "  không đủ ý nghĩa thống kê.", ""]
if SMOKE:
    lines += ["", "> ⚠️ **ĐÂY LÀ KẾT QUẢ CHẠY THỬ — số liệu vô nghĩa.**", ""]

(RESULTS / "summary.md").write_text("\n".join(lines), encoding="utf-8")
df.to_csv(RESULTS / "key_results.csv", index=False)
print("da tao summary.md va key_results.csv")

da tao summary.md va key_results.csv


In [25]:
_zip = shutil.make_archive(str(OUT / "kd_results"), "zip", root_dir=RESULTS)
print("dong goi:", _zip, f"({Path(_zip).stat().st_size/1024**2:.1f} MB)")
print("\nKet qua:")
for p in sorted(RESULTS.rglob("*")):
    if p.is_file():
        print("   ", p.relative_to(RESULTS))
print("\nTrong so:")
for s, d in STAGE_DIR.items():
    b = RUNS / s.lower() / "weights" / "best.pt"
    b = b if b.exists() else RUNS / "kd" / "weights" / "best.pt"
    if b.exists():
        print(f"   {s:9s} {b}  ({b.stat().st_size/1024**2:.2f} MB)")

if SMOKE:
    print("\n" + "*" * 66)
    print("*  CHAY THU HOAN TAT — ca 3 giai doan train, export va danh gia deu chay duoc.")
    print("*  Gio doi:  RUN_MODE = 'FULL_TRAIN'  va  CONFIRM_FULL_TRAIN = True")
    print("*" * 66)
    (OUT / "smoke_test_passed.json").write_text(json.dumps({
        "passed": True, "time": time.strftime("%Y-%m-%d %H:%M:%S"),
        "stages": ["teacher", "baseline", "kd", "export", "eval"],
        "n_configs_evaluated": len(all_metrics)}, indent=2))

dong goi: /kaggle/working/kd_outputs/full/kd_results.zip (0.1 MB)

Ket qua:
    key_results.csv
    metrics/data_leakage.json
    metrics/onnx_check.json
    metrics/thay_yolo26s.json
    metrics/tro_chung_cat.json
    metrics/tro_chung_cat_onnx_fp32.json
    metrics/tro_chung_cat_onnx_int8.json
    metrics/tro_doi_chung.json
    plots/accuracy_and_size.png
    plots/distillation_loss.png
    plots/kd_vs_baseline.png
    summary.md
    tables/kd_comparison.csv
    tables/key_comparisons.csv
    tables/per_class_ap.csv
    tables/threshold_checks.csv

Trong so:
   TEACHER   /kaggle/working/kd_outputs/full/runs/teacher/weights/best.pt  (19.39 MB)
   BASELINE  /kaggle/working/kd_outputs/full/runs/baseline/weights/best.pt  (5.15 MB)
   KD        /kaggle/working/kd_outputs/full/runs/kd/weights/best.pt  (5.15 MB)


---
## Cách đọc kết quả

**1. Phép so sánh quan trọng nhất: trò chưng cất so với trò đối chứng.** Hai mô hình có đúng
cùng kiến trúc, cùng số tham số, cùng tốc độ, cùng ngân sách huấn luyện. Chênh lệch duy nhất
đến từ chưng cất. Nếu `Δ mAP` dương, đó là **độ chính xác tăng thêm miễn phí** — không tốn một
chút chi phí nào lúc suy luận.

**2. Cột `Δ AP nhỏ` đáng chú ý riêng.** Chưng cất truyền đặc trưng từ ba tầng cổ mạng, nơi
thông tin đa tỷ lệ được tổng hợp — nếu nó giúp, khả năng cao là giúp nhiều nhất ở vật thể nhỏ.

**3. Nếu `Δ` gần bằng không, đó vẫn là kết quả.** Bộ dữ liệu này nhỏ và tương đối dễ; mô hình
nền đã đạt mAP cao nên khoảng trống cải thiện rất hẹp. Kết luận trung thực trong trường hợp đó
là: *"chưng cất không mang lại cải thiện đáng kể vì mô hình học trò đã gần chạm trần của bộ dữ
liệu"* — hoàn toàn hợp lệ để báo cáo, và có ích hơn nhiều so với việc chỉnh ngưỡng cho vừa số.

**4. Đừng gộp dòng GPU với dòng CPU khi tính tăng tốc.** Cột `Thiết bị` ghi rõ từng dòng chạy
ở đâu.